In [2]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


22450
4182


In [3]:
full_data = concatenate_datasets([train_data, test_data])

In [4]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [5]:
eval_examples = [{
    "title" : row["title"],
    "question": row["question"],
    "answers": row["answers"]["text"],
}
for row in full_data]

print(len(eval_examples))
print(eval_examples[0])

26632
{'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'answers': ['DISTRIBUTOR AGREEMENT']}


In [6]:
from collections import defaultdict

grouped = defaultdict(list)
for ex in eval_examples:
    key = (ex['title'], ex['question'])
    grouped[key].extend(ex['answers'])

eval_dataset_raw = [
    {"title": title, "question": question, "ground_truth": answers}
    for (title, question), answers in grouped.items() if len(answers)>0
]

print(f"Number of (contract, question) pairs with at least one answer: {len(eval_dataset_raw)}")

Number of (contract, question) pairs with at least one answer: 6702


In [7]:
import pandas as pd

def get_category(question):
    if '"' in question:
        return question.split('"')[1]
    return None

for item in eval_dataset_raw:
    item["category"]= get_category(item['question'])

by_category = defaultdict(list)
for item in eval_dataset_raw:
    by_category[item['category']].append(item)

rows= []
for category, items in by_category.items():
    lengths = [len(" ".join(it["ground_truth"])) for it in items]
    rows.append({
        "category": category,
        "n_examples": len(items),
        "avg_length_chars": sum(lengths)/len(lengths),
        "min_length_chars": min(lengths),
        "max_length_chars": max(lengths)
    })

summary_df = pd.DataFrame(rows).sort_values("avg_length_chars")
print(summary_df.to_string(index=False))

                          category  n_examples  avg_length_chars  min_length_chars  max_length_chars
                    Agreement Date         470         18.442553                 3               121
                     Document Name         510         30.482353                11               285
                    Effective Date         390         71.394872                 3               717
                           Parties         509        122.280943                17              1448
                     Governing Law         437        219.004577                42               885
                   Expiration Date         413        260.990315                12              1777
           Third Party Beneficiary          32        285.406250                64              1030
       Termination For Convenience         183        298.672131                35              2063
Notice Period To Terminate Renewal         111        337.279279               133         

In [8]:
no_answer_dataset = [
    {"title": title, "question": question, "category": get_category(question)}
    for (title, question), answers in grouped.items() if len(answers)==0
]

print(f"Number of (contract, question) pairs with No answer (clause absent): {len(no_answer_dataset)}")

Number of (contract, question) pairs with No answer (clause absent): 14208


In [9]:
import random
random.seed(42)

N_POSITIVE_PER_CATEGORY = 5
N_NEGATIVE_PER_CATEGORY = 3

positive_sample = []
for category, items in by_category.items():
    n = min(N_POSITIVE_PER_CATEGORY, len(items))
    positive_sample.extend(random.sample(items, n))

print(f"positive sample size: {len(positive_sample)} (across {len(by_category)} categories)")

no_answer_by_category = defaultdict(list)
for item in no_answer_dataset:
    no_answer_by_category[item["category"]].append(item)

negative_sample = []
for category, items in no_answer_by_category.items():
    n = min(N_NEGATIVE_PER_CATEGORY, len(items))
    negative_sample.extend(random.sample(items, n))

print(f"negative sample size: {len(negative_sample)} (across {len(no_answer_by_category)} categories)")


positive sample size: 205 (across 41 categories)
negative sample size: 118 (across 40 categories)


In [10]:
positive_sample[0]

{'title': 'EmbarkComInc_19991008_S-1A_EX-10.10_6487661_EX-10.10_Co-Branding Agreement',
 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
 'ground_truth': ['CO-BRANDING AGREEMENT'],
 'category': 'Document Name'}

In [11]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

In [12]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

In [13]:
for item in positive_sample:
    item["ground_truth_clean"] = [clean_contract_text(a) for a in item["ground_truth"]]

In [14]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore

def build_index_with_embedding(embed_model, nodes, collection_name, qdrant_client):
    vector_store = QdrantVectorStore(client = qdrant_client, collection_name=collection_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    return VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

In [16]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from collections import defaultdict

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

Splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    tokenizer=tokenizer.encode
)

documents = [
    Document(
        text=c["clean_context"], metadata={"title": c["title"]})
        for c in unique_contracts
]

nodes = Splitter.get_nodes_from_documents(documents)

# chunk index for each contract
counters = defaultdict(int)
for n in nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"Number of contracts : {len(documents)}")
print(f"Number of chunks: {len(nodes)}")
print(f"Average chunks per contract: {len(nodes)/len(documents):.1f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7558 > 512). Running this sequence through the model will result in indexing errors


Number of contracts : 510
Number of chunks: 14230
Average chunks per contract: 27.9


In [17]:
from qdrant_client import QdrantClient

client = QdrantClient(path="../artifacts/qdrant_storage")

In [22]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.ollama import OllamaEmbedding

bge_large_embed = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")
index_bge_large = build_index_with_embedding(bge_large_embed, nodes, "cuad_bge_large", client)

nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")
index_nomic = build_index_with_embedding(nomic_embed, nodes, "cuad_nomic", client)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6145.19it/s]


In [18]:
category_to_tier = {}

for _, row in summary_df.iterrows():
    if row["avg_length_chars"] < 150:
        tier = 1
    elif row['avg_length_chars'] < 900:
        tier = 2
    else:
        tier = 3
    
    category_to_tier[row['category']] = tier

print(category_to_tier)

{'Agreement Date': 1, 'Document Name': 1, 'Effective Date': 1, 'Parties': 1, 'Governing Law': 2, 'Expiration Date': 2, 'Third Party Beneficiary': 2, 'Termination For Convenience': 2, 'Notice Period To Terminate Renewal': 2, 'Renewal Term': 2, 'Anti-Assignment': 2, 'No-Solicit Of Customers': 2, 'Volume Restriction': 2, 'No-Solicit Of Employees': 2, 'Most Favored Nation': 2, 'Price Restrictions': 2, 'Uncapped Liability': 2, 'Covenant Not To Sue': 2, 'Non-Disparagement': 2, 'Unlimited/All-You-Can-Eat-License': 2, 'Liquidated Damages': 2, 'Warranty Duration': 2, 'Competitive Restriction Exception': 2, 'Change Of Control': 2, 'Exclusivity': 2, 'Joint Ip Ownership': 2, 'Non-Compete': 2, 'Revenue/Profit Sharing': 2, 'Minimum Commitment': 3, 'Non-Transferable License': 3, 'Cap On Liability': 3, 'Audit Rights': 3, 'Ip Ownership Assignment': 3, 'Insurance': 3, 'Affiliate License-Licensee': 3, 'Post-Termination Services': 3, 'License Grant': 3, 'Irrevocable Or Perpetual License': 3, 'Rofr/Rofo/Ro

In [19]:
category_to_tier["Agreement Date"] = 2
category_to_tier["Effective Date"] = 2

In [25]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

def retrieve_scoped_generic(title, question, category, index, top_k=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier==1:
        contract_nodes = sorted(contract_nodes, key=lambda n:n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [n.get_content() for n in selected]
    
    filtered_retriever = index.as_retriever(
        similarity_top_k=top_k,
        filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    return [n.get_content() for n in retrieved_nodes]

In [28]:
import re
import pandas as pd

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def is_hit(reference_answers, retrieved_chunks):
    normalized_chunks = [normalize_text(chunk) for chunk in retrieved_chunks]
    for ref in reference_answers:
        normalized_ref = normalize_text(ref)
        if not normalized_ref:
            continue
        for chunk in normalized_chunks:
            if normalized_ref in chunk:
                return 1
        
    return 0

def compute_hit_rate(sample, index, top_k=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        reference_answers = item["ground_truth"]

        retrieved_chunks = retrieve_scoped_generic(title, question, category, index, top_k=top_k)
        hit = is_hit(reference_answers, retrieved_chunks)

        results.append({"title": title, "category": category, "question": question, "hit": hit})

    
    df_hits = pd.DataFrame(results)
    overall_hit_rate = df_hits["hit"].mean()
    hit_rate_by_category = df_hits.groupby("category")["hit"].mean().sort_values()
    return overall_hit_rate, hit_rate_by_category, df_hits
    

In [28]:
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index_bge_small = build_index_with_embedding(embed_model, nodes, "cuad_bge_small", client)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5050.38it/s]


In [29]:
baseline_hit_rate, baseline_hit_vy_category, baseline_df = compute_hit_rate(positive_sample, index_bge_small)
print(f"Baseline (bge -small) hit rate: {baseline_hit_rate:.3f}")

bge_large_hit_rate, bge_large_hit_vy_category, bge_large_df = compute_hit_rate(positive_sample, index_bge_large)
print(f"Baseline (bge -large) hit rate: {bge_large_hit_rate:.3f}")

nomic_hit_rate, nomic_hit_vy_category, nomic_df = compute_hit_rate(positive_sample, index_nomic)
print(f"Baseline (nomic) hit rate: {nomic_hit_rate:.3f}")

Baseline (bge -small) hit rate: 0.498
Baseline (bge -large) hit rate: 0.546
Baseline (nomic) hit rate: 0.561


In [30]:
baseline_hit_vy_category

category
Unlimited/All-You-Can-Eat-License     0.0
Covenant Not To Sue                   0.0
Minimum Commitment                    0.2
Agreement Date                        0.2
Rofr/Rofo/Rofn                        0.2
Revenue/Profit Sharing                0.2
Post-Termination Services             0.2
Competitive Restriction Exception     0.2
Non-Transferable License              0.2
Non-Disparagement                     0.2
No-Solicit Of Customers               0.2
Irrevocable Or Perpetual License      0.2
Ip Ownership Assignment               0.2
Price Restrictions                    0.4
Most Favored Nation                   0.4
Volume Restriction                    0.4
Affiliate License-Licensee            0.4
Affiliate License-Licensor            0.4
Effective Date                        0.4
Joint Ip Ownership                    0.6
Uncapped Liability                    0.6
Third Party Beneficiary               0.6
Termination For Convenience           0.6
Audit Rights             

In [31]:
bge_large_hit_vy_category

category
Covenant Not To Sue                   0.0
Revenue/Profit Sharing                0.2
Non-Transferable License              0.2
Non-Disparagement                     0.2
No-Solicit Of Customers               0.2
Competitive Restriction Exception     0.2
Most Favored Nation                   0.2
Volume Restriction                    0.2
Affiliate License-Licensee            0.4
Unlimited/All-You-Can-Eat-License     0.4
Post-Termination Services             0.4
Liquidated Damages                    0.4
Irrevocable Or Perpetual License      0.4
Ip Ownership Assignment               0.4
Minimum Commitment                    0.4
Governing Law                         0.4
Affiliate License-Licensor            0.4
Agreement Date                        0.4
Effective Date                        0.4
Audit Rights                          0.6
Cap On Liability                      0.6
Non-Compete                           0.6
Warranty Duration                     0.6
Expiration Date          

In [32]:
nomic_hit_vy_category

category
Most Favored Nation                   0.0
Covenant Not To Sue                   0.0
Irrevocable Or Perpetual License      0.0
Affiliate License-Licensor            0.2
No-Solicit Of Customers               0.2
Volume Restriction                    0.2
Effective Date                        0.2
Affiliate License-Licensee            0.4
Revenue/Profit Sharing                0.4
Price Restrictions                    0.4
Non-Disparagement                     0.4
Non-Compete                           0.4
Liquidated Damages                    0.4
Ip Ownership Assignment               0.4
Minimum Commitment                    0.4
Warranty Duration                     0.4
Exclusivity                           0.4
Competitive Restriction Exception     0.4
Agreement Date                        0.4
Expiration Date                       0.6
Joint Ip Ownership                    0.6
License Grant                         0.6
Unlimited/All-You-Can-Eat-License     0.6
Audit Rights             

In [33]:
merged = baseline_df[["title", "question", "category", "hit"]].rename(columns={"hit": "hit_bge_small"})
merged = merged.merge(
    bge_large_df[["title", "question", "hit"]].rename(columns={"hit": "hit_bge_large"}),
    on=["title", "question"]
)
merged = merged.merge(
    nomic_df[["title", "question", "hit"]].rename(columns={"hit": "hit_nomic"}),
    on=["title", "question"]
)

merged["n_hits"] = merged[["hit_bge_small", "hit_bge_large", "hit_nomic"]].sum(axis=1)

persistent_miss = (merged["n_hits"] == 0).mean()
persistent_hit = (merged["n_hits"] == 3).mean()
mixed = ((merged["n_hits"] > 0) & (merged["n_hits"] < 3)).mean()

print(f"Echec avec les 3 modeles (probleme structurel, pas l'embedding) : {persistent_miss:.1%}")
print(f"Succes avec les 3 modeles : {persistent_hit:.1%}")
print(f"Resultat mixte (sensible au choix du modele) : {mixed:.1%}")


Echec avec les 3 modeles (probleme structurel, pas l'embedding) : 30.7%
Succes avec les 3 modeles : 37.1%
Resultat mixte (sensible au choix du modele) : 32.2%


In [36]:
e5_embed = HuggingFaceEmbedding(
    model_name="intfloat/e5-large-v2",
    query_instruction="query: ",
    text_instruction="passage: ",
    embed_batch_size=256)

index_e5 = build_index_with_embedding(e5_embed, nodes, "cuad_e5_large", client)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4017.72it/s]


In [37]:
e5_hit_rate, e5_hit_by_category, e5_df = compute_hit_rate(positive_sample, index_e5)
print(f"e5-large-v2 hit rate: {e5_hit_rate:.3f}")

e5-large-v2 hit rate: 0.541


In [38]:
import random

failure_pool = merged[(merged["n_hits"] == 0) & (~merged["category"].isin(["Document Name", "Parties"]))]

random.seed(123)
sample_indices = random.sample(list(failure_pool.index), 10)
inspection_sample = failure_pool.loc[sample_indices]

positive_lookup = {(item["title"], item["question"]): item for item in positive_sample}

In [42]:
inspection_sample

,title,question,category,hit_bge_small,hit_bge_large,hit_nomic,n_hits
18,ChinaRealEstateInformationCorp_20090929_F-1_EX...,Highlight the parts (if any) of this contract ...,Effective Date,0,0,0,0
69,"Array BioPharma Inc. - LICENSE, DEVELOPMENT AN...",Highlight the parts (if any) of this contract ...,Minimum Commitment,0,0,0,0
21,"Magenta Therapeutics, Inc. - Master Developmen...",Highlight the parts (if any) of this contract ...,Expiration Date,0,0,0,0
169,"Sonos, Inc. - Manufacturing Agreement",Highlight the parts (if any) of this contract ...,Non-Transferable License,0,0,0,0
94,ReedsInc_20191113_10-Q_EX-10.4_11888303_EX-10....,Highlight the parts (if any) of this contract ...,Covenant Not To Sue,0,0,0,0
202,PapaJohnsInternationalInc_20190617_8-K_EX-10.1...,Highlight the parts (if any) of this contract ...,Non-Disparagement,0,0,0,0
22,GOCALLINC_03_30_2000-EX-10.7-Promotion Agreement,Highlight the parts (if any) of this contract ...,Expiration Date,0,0,0,0
179,TheglobeComInc_19990503_S-1A_EX-10.20_5416126_...,Highlight the parts (if any) of this contract ...,Most Favored Nation,0,0,0,0
17,GLOBALTECHNOLOGIESLTD_06_08_2020-EX-10.16-CONS...,Highlight the parts (if any) of this contract ...,Effective Date,0,0,0,0
92,PHLVARIABLEINSURANCECOCT_08_17_2009-EX-10.1-ST...,Highlight the parts (if any) of this contract ...,Covenant Not To Sue,0,0,0,0


In [43]:
for _, row in inspection_sample.iterrows():
    title = row["title"]
    question = row["question"]
    category = row["category"]
    item = positive_lookup[(title, question)]
    ground_truth = item["ground_truth"]

    retrieved_chunks = retrieve_scoped_generic(title, question, category, index_nomic, top_k=3)

    print("=" * 100)
    print(f"Categorie: {category}")
    print(f"Question: {question}")
    print(f"Ground truth: {ground_truth}")
    print("-" * 100)
    for i, chunk in enumerate(retrieved_chunks):
        print(f"[Chunk {i}]\n{chunk}\n")

Categorie: Effective Date
Question: Highlight the parts (if any) of this contract related to "Effective Date" that should be reviewed by a lawyer. Details: The date when the contract is effective
Ground truth: ['"Effective Date" means the Closing Date as set forth in the Share Purchase Agreement.']
----------------------------------------------------------------------------------------------------
[Chunk 0]
ARTICLE VI TERM AND TERMINATION

6.1. Term. The initial term of this Agreement (the "Initial Term") shall commence on the Effective Date and shall continue for a period of ten (10) years thereafter. Beginning twelve (12) months prior to the expiration of the Initial Term, the Parties shall use

7

Source: CHINA REAL ESTATE INFORMATION CORP, F-1, 9/29/2009

good faith efforts to negotiate an extension of the term of this Agreement (the Initial Term together with any applicable extension, the "Term").

6.2. Termination for Bankruptcy. Either Party may immediately terminate this Agreem

In [45]:
vector_store_hybrid = QdrantVectorStore(
    collection_name = "cuad_nomic_bm25_hybrid",
    client = client,
    enable_hybrid = True,
    fastembed_sparse_model = "Qdrant/bm25"
)
storage_context_hybrid = StorageContext.from_defaults(vector_store=vector_store_hybrid)
index_hybrid = VectorStoreIndex(nodes, storage_context=storage_context_hybrid, embed_model=nomic_embed)


In [47]:
def retrieve_scoped_hybrid(title, question, category, index, top_k=3, sparse_top_k=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n:n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [n.get_content() for n in selected]
    
    filtered_retriever = index.as_retriever(
        similarity_top_k=top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    return [n.get_content() for n in retrieved_nodes]

In [48]:
def compute_hit_rate(sample, index, retrieve_fn=retrieve_scoped_generic, top_k=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        reference_answers = item["ground_truth"]

        retrieved_chunks = retrieve_fn(title, question, category, index, top_k=top_k)
        hit = is_hit(reference_answers, retrieved_chunks)

        results.append({"title": title, "category": category, "question": question, "hit": hit})

    
    df_hits = pd.DataFrame(results)
    overall_hit_rate = df_hits["hit"].mean()
    hit_rate_by_category = df_hits.groupby("category")["hit"].mean().sort_values()
    return overall_hit_rate, hit_rate_by_category, df_hits

In [49]:
hybrid_hit_rate, hybrid_hit_by_category, hybrid_df = compute_hit_rate(
    positive_sample, index_hybrid, retrieve_fn=retrieve_scoped_hybrid)
print(f"Hybrid (nomic + BM25) hit rate: {hybrid_hit_rate:.3f}")

Hybrid (nomic + BM25) hit rate: 0.610


In [52]:
hybrid_hit_by_category

category
Effective Date                        0.0
Minimum Commitment                    0.2
Revenue/Profit Sharing                0.2
Most Favored Nation                   0.2
Volume Restriction                    0.2
Competitive Restriction Exception     0.2
Covenant Not To Sue                   0.2
Exclusivity                           0.2
Price Restrictions                    0.4
Non-Disparagement                     0.4
Non-Compete                           0.4
Liquidated Damages                    0.4
Irrevocable Or Perpetual License      0.4
Ip Ownership Assignment               0.4
Warranty Duration                     0.4
Affiliate License-Licensor            0.4
Agreement Date                        0.4
Affiliate License-Licensee            0.6
Joint Ip Ownership                    0.6
Termination For Convenience           0.6
Cap On Liability                      0.6
No-Solicit Of Customers               0.6
Rofr/Rofo/Rofn                        0.6
Non-Transferable License 

In [56]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(
    model = "BAAI/bge-reranker-base",
    top_n = 3
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9949.43it/s]


In [25]:
def retrieve_scoped_reranked(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [26]:
def compute_hit_rate_reranked(sample, index, retrieve_fn=retrieve_scoped_reranked, rerank_top_n=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        reference_answers = item["ground_truth"]

        retrieved = retrieve_fn(title, question, category, index, rerank_top_n=rerank_top_n)
        retrieved_texts = [r["text"] for r in retrieved]
        retrieved_scores = [r["score"] for r in retrieved]

        hit = is_hit(reference_answers, retrieved_texts)

        results.append({
            "title": title,
            "category": category,
            "question": question,
            "hit": hit,
            "retrieved_chunks": retrieved_texts,
            "retrieved_scores": retrieved_scores,
        })

    df_hits = pd.DataFrame(results)
    overall_hit_rate = df_hits["hit"].mean()
    hit_rate_by_category = df_hits.groupby("category")["hit"].mean().sort_values()
    return overall_hit_rate, hit_rate_by_category, df_hits


In [64]:
reranked_hit_rate, reranked_hit_by_category, reranked_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid
)
print(f"Hybrid + reranking hit rate: {reranked_hit_rate:.3f}")


Hybrid + reranking hit rate: 0.585


In [65]:
reranked_hit_by_category

category
Minimum Commitment                    0.0
Agreement Date                        0.0
Most Favored Nation                   0.0
Rofr/Rofo/Rofn                        0.0
Revenue/Profit Sharing                0.0
Ip Ownership Assignment               0.2
No-Solicit Of Customers               0.2
Competitive Restriction Exception     0.2
Volume Restriction                    0.2
Expiration Date                       0.4
Covenant Not To Sue                   0.4
Price Restrictions                    0.4
Liquidated Damages                    0.4
Non-Compete                           0.6
Non-Transferable License              0.6
Termination For Convenience           0.6
Uncapped Liability                    0.6
Non-Disparagement                     0.6
Warranty Duration                     0.6
Joint Ip Ownership                    0.6
Exclusivity                           0.6
Effective Date                        0.6
Audit Rights                          0.6
License Grant            

In [66]:
reranked_merged = hybrid_df[["title", "question", "category", "hit"]].rename(columns={"hit": "hit_hybrid"})
reranked_merged = reranked_merged.merge(
    reranked_df[["title", "question", "hit"]].rename(columns={"hit": "hit_reranked"}),
    on=["title", "question"]
)

regressions = reranked_merged[(reranked_merged["hit_hybrid"] == 1) & (reranked_merged["hit_reranked"] == 0)]
print(f"Nombre de regressions (hit avec hybride seul, miss apres reranking) : {len(regressions)}")
regressions


Nombre de regressions (hit avec hybride seul, miss apres reranking) : 21


,title,question,category,hit_hybrid,hit_reranked
10,"VNUE,INC_07_10_2015-EX-10.1-SPONSORSHIP AGREEMENT",Highlight the parts (if any) of this contract ...,Agreement Date,1,0
12,LOOKSMARTLTD_07_20_2012-EX-99.(D)(I)-SPONSORSH...,Highlight the parts (if any) of this contract ...,Agreement Date,1,0
21,"Magenta Therapeutics, Inc. - Master Developmen...",Highlight the parts (if any) of this contract ...,Expiration Date,1,0
22,GOCALLINC_03_30_2000-EX-10.7-Promotion Agreement,Highlight the parts (if any) of this contract ...,Expiration Date,1,0
23,"VNUE,INC_07_10_2015-EX-10.1-SPONSORSHIP AGREEMENT",Highlight the parts (if any) of this contract ...,Expiration Date,1,0
43,"Loop Industries, Inc. - Marketing Agreement",Highlight the parts (if any) of this contract ...,No-Solicit Of Customers,1,0
44,WPPPLC_04_30_2020-EX-4.28-SERVICE AGREEMENT,Highlight the parts (if any) of this contract ...,No-Solicit Of Customers,1,0
46,BUFFALOWILDWINGSINC_06_05_1998-EX-10.3-FRANCHI...,Highlight the parts (if any) of this contract ...,No-Solicit Of Employees,1,0
50,STAMPSCOMINC_06_24_1999-EX-10.18-SPONSORSHIP A...,Highlight the parts (if any) of this contract ...,Rofr/Rofo/Rofn,1,0
51,PhoenixNewMediaLtd_20110421_F-1_EX-10.17_69583...,Highlight the parts (if any) of this contract ...,Rofr/Rofo/Rofn,1,0


In [67]:
for _, row in regressions.head(5).iterrows():
    title = row["title"]
    question = row["question"]
    category = row["category"]
    item = positive_lookup[(title, question)]
    ground_truth = item["ground_truth"]

    pool_10 = retrieve_scoped_hybrid(title, question, category, index_hybrid, top_k=10, sparse_top_k=10)
    gt_in_pool = is_hit(ground_truth, pool_10)

    reranked_result = retrieve_scoped_reranked(title, question, category, index_hybrid, rerank_top_n=3)

    print("=" * 100)
    print(f"Categorie: {category}")
    print(f"Question: {question}")
    print(f"Ground truth: {ground_truth}")
    print(f"Bon chunk present dans le pool top-10 avant reranking ? {'OUI' if gt_in_pool else 'NON'}")
    print("-" * 100)
    print("Top 3 APRES reranking (avec scores) :")
    for r in reranked_result:
        print(f"  score={r['score']}: {r['text'][:150]}...")


Categorie: Agreement Date
Question: Highlight the parts (if any) of this contract related to "Agreement Date" that should be reviewed by a lawyer. Details: The date of the contract
Ground truth: ['June 23, 2015']
Bon chunk present dans le pool top-10 avant reranking ? OUI
----------------------------------------------------------------------------------------------------
Top 3 APRES reranking (avec scores) :
  score=0.013447846285998821: Page of 7 of Sponsorship Agreement

17. Governing Law and Venue. This Agreement is to be governed and construed according to the laws of the State of ...
  score=0.002166120568290353: Page of 6 of Sponsorship Agreement

13. Notices. All notices and payment given in accordance with this Agreement will be effective if hand delivered o...
  score=0.0008895997307263315: The Sponsor must provide the consideration in Section 2 regardless of whether the Event is actually held. 9. Warranties. (a) The Sponsor's Warranty. T...
Categorie: Agreement Date
Question:

In [33]:
from llama_index.core.postprocessor import SentenceTransformerRerank
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-v2-m3",
    top_n=3
)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5051.01it/s]


In [71]:
import pickle

checkpoint = {
    "baseline_hit_rate": baseline_hit_rate,
    "baseline_hit_vy_category": baseline_hit_vy_category,
    "baseline_df": baseline_df,
    "bge_large_hit_rate": bge_large_hit_rate,
    "bge_large_hit_vy_category": bge_large_hit_vy_category,
    "bge_large_df": bge_large_df,
    "nomic_hit_rate": nomic_hit_rate,
    "nomic_hit_vy_category": nomic_hit_vy_category,
    "nomic_df": nomic_df,
    "e5_hit_rate": e5_hit_rate,
    "e5_hit_by_category": e5_hit_by_category,
    "e5_df": e5_df,
    "hybrid_hit_rate": hybrid_hit_rate,
    "hybrid_hit_by_category": hybrid_hit_by_category,
    "hybrid_df": hybrid_df,
    "reranked_hit_rate": reranked_hit_rate,
    "reranked_hit_by_category": reranked_hit_by_category,
    "reranked_df": reranked_df,
    "merged": merged,
    "positive_sample": positive_sample,
    "negative_sample": negative_sample,
    "category_to_tier": category_to_tier,
}

with open("../artifacts/baseline_v2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Checkpoint sauvegarde.")


Checkpoint sauvegarde.


In [1]:
import pickle

with open("../artifacts/baseline_v2_checkpoint.pkl", "rb") as f:
    checkpoint = pickle.load(f)

baseline_hit_rate = checkpoint["baseline_hit_rate"]
baseline_hit_vy_category = checkpoint["baseline_hit_vy_category"]
baseline_df = checkpoint["baseline_df"]
bge_large_hit_rate = checkpoint["bge_large_hit_rate"]
bge_large_hit_vy_category = checkpoint["bge_large_hit_vy_category"]
bge_large_df = checkpoint["bge_large_df"]
nomic_hit_rate = checkpoint["nomic_hit_rate"]
nomic_hit_vy_category = checkpoint["nomic_hit_vy_category"]
nomic_df = checkpoint["nomic_df"]
e5_hit_rate = checkpoint["e5_hit_rate"]
e5_hit_by_category = checkpoint["e5_hit_by_category"]
e5_df = checkpoint["e5_df"]
hybrid_hit_rate = checkpoint["hybrid_hit_rate"]
hybrid_hit_by_category = checkpoint["hybrid_hit_by_category"]
hybrid_df = checkpoint["hybrid_df"]
reranked_hit_rate = checkpoint["reranked_hit_rate"]
reranked_hit_by_category = checkpoint["reranked_hit_by_category"]
reranked_df = checkpoint["reranked_df"]
merged = checkpoint["merged"]
positive_sample = checkpoint["positive_sample"]
negative_sample = checkpoint["negative_sample"]
category_to_tier = checkpoint["category_to_tier"]

print("Checkpoint recharge.")


Checkpoint recharge.


In [21]:
from llama_index.embeddings.ollama import OllamaEmbedding
nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")

In [23]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext
vector_store_hybrid = QdrantVectorStore(
    collection_name="cuad_nomic_bm25_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
index_hybrid = VectorStoreIndex.from_vector_store(vector_store_hybrid, embed_model=nomic_embed)


In [30]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter

In [34]:
reranked_m3_hit_rate, reranked_m3_hit_by_category, reranked_m3_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid
)
print(f"Hybrid + reranking hit rate: {reranked_m3_hit_rate:.3f}")


Hybrid + reranking hit rate: 0.644


In [36]:
reranked_m3_hit_by_category

category
Minimum Commitment                    0.0
Most Favored Nation                   0.0
Revenue/Profit Sharing                0.0
Rofr/Rofo/Rofn                        0.2
Competitive Restriction Exception     0.2
Volume Restriction                    0.4
Expiration Date                       0.4
No-Solicit Of Customers               0.4
Non-Compete                           0.4
Covenant Not To Sue                   0.4
Ip Ownership Assignment               0.4
Liquidated Damages                    0.4
Post-Termination Services             0.6
Price Restrictions                    0.6
Termination For Convenience           0.6
Unlimited/All-You-Can-Eat-License     0.6
Non-Disparagement                     0.6
Warranty Duration                     0.6
Affiliate License-Licensor            0.6
Insurance                             0.6
Agreement Date                        0.6
Exclusivity                           0.6
Effective Date                        0.6
License Grant            

In [38]:
negative_sample[0]

{'title': 'VERTICALNETINC_04_01_2002-EX-10.19-MAINTENANCE AND SUPPORT AGREEMENT',
 'question': 'Highlight the parts (if any) of this contract related to "Notice Period To Terminate Renewal" that should be reviewed by a lawyer. Details: What is the notice period required to terminate renewal?',
 'category': 'Notice Period To Terminate Renewal'}

In [39]:
def compute_scores_for_sample(sample, index, retrieve_fn=retrieve_scoped_reranked, rerank_top_n=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]

        retrieved = retrieve_fn(title, question, category, index, rerank_top_n=rerank_top_n)
        scores = [r["score"] for r in retrieved if r["score"] is not None]
        top_score = max(scores) if scores else None

        results.append({
            "title": title,
            "category": category,
            "top_score": top_score,
        })

    return pd.DataFrame(results)

negative_scores_df = compute_scores_for_sample(negative_sample, index_hybrid)


In [40]:
positive_scores_df = reranked_m3_df[["title", "category"]].copy()
positive_scores_df["top_score"] = reranked_m3_df["retrieved_scores"].apply(
    lambda scores: max([s for s in scores if s is not None], default=None)
)


In [41]:
print("Positifs (clause presente) - distribution du meilleur score :")
print(positive_scores_df["top_score"].describe())

print("\nNegatifs (clause absente) - distribution du meilleur score :")
print(negative_scores_df["top_score"].describe())


Positifs (clause presente) - distribution du meilleur score :
count    195.000000
mean       0.160974
std        0.231528
min        0.002477
25%        0.015265
50%        0.046863
75%        0.194767
max        0.890346
Name: top_score, dtype: float64

Negatifs (clause absente) - distribution du meilleur score :
count    117.000000
mean       0.036352
std        0.067611
min        0.000220
25%        0.007452
50%        0.014016
75%        0.037409
max        0.517739
Name: top_score, dtype: float64


In [42]:
import re

def strip_boilerplate(question):
    match = re.search(r'"([^"]+)"', question)
    category_quoted = match.group(0) if match else ""
    details = question.split("Details: ", 1)[1] if "Details: " in question else ""
    return f"{category_quoted}. {details}".strip()

def retrieve_scoped_reranked_short_q(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    filtered_retriever = index.as_retriever(
        similarity_top_k = retrieve_top_k,
        sparse_too_k = sparse_top_k,
        vector_store_query_mode = "hybrid",
        filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)]) 
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text":n.get_content(), "score":n.score} for n in reranked_nodes]

In [44]:
from llama_index.core.postprocessor import SentenceTransformerRerank

reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-large", top_n=3)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6264.75it/s]


In [45]:

reranked_large_short_hit_rate, reranked_large_short_hit_by_category, reranked_large_short_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid, retrieve_fn=retrieve_scoped_reranked_short_q
)
print(f"Hybrid + bge-reranker-large (514 tok) + question raccourcie: {reranked_large_short_hit_rate:.3f}")

Hybrid + bge-reranker-large (514 tok) + question raccourcie: 0.688


In [46]:
reranked_large_short_hit_by_category

category
Minimum Commitment                    0.0
Revenue/Profit Sharing                0.0
Most Favored Nation                   0.2
Liquidated Damages                    0.4
Rofr/Rofo/Rofn                        0.4
Competitive Restriction Exception     0.4
Covenant Not To Sue                   0.4
Exclusivity                           0.4
Ip Ownership Assignment               0.4
Termination For Convenience           0.6
No-Solicit Of Customers               0.6
Non-Compete                           0.6
Non-Disparagement                     0.6
Non-Transferable License              0.6
Post-Termination Services             0.6
Unlimited/All-You-Can-Eat-License     0.6
Volume Restriction                    0.6
Price Restrictions                    0.6
Affiliate License-Licensor            0.6
Effective Date                        0.6
Source Code Escrow                    0.8
Notice Period To Terminate Renewal    0.8
Affiliate License-Licensee            0.8
Warranty Duration        

In [47]:
negative_large_short_scores_df = compute_scores_for_sample(negative_sample, index_hybrid)

In [49]:
positive_large_short_scores_df = reranked_large_short_df[["title", "category"]].copy()
positive_large_short_scores_df["top_score"] = reranked_large_short_df["retrieved_scores"].apply(
    lambda scores: max([s for s in scores if s is not None], default=None)
)

In [50]:
print("Positifs (clause presente) - distribution du meilleur score :")
print(positive_large_short_scores_df["top_score"].describe())

print("\nNegatifs (clause absente) - distribution du meilleur score :")
print(negative_large_short_scores_df["top_score"].describe())

Positifs (clause presente) - distribution du meilleur score :
count    195.000000
mean       0.053256
std        0.125090
min        0.000100
25%        0.000731
50%        0.003633
75%        0.033293
max        0.786843
Name: top_score, dtype: float64

Negatifs (clause absente) - distribution du meilleur score :
count    117.000000
mean       0.003114
std        0.007262
min        0.000098
25%        0.000548
50%        0.000978
75%        0.002355
max        0.054334
Name: top_score, dtype: float64


In [51]:
current_failures = reranked_large_short_df[reranked_large_short_df["hit"] == 0]
print(f"Echecs avec le pipeline actuel : {len(current_failures)} / {len(reranked_large_short_df)}")
print(current_failures["category"].value_counts())


Echecs avec le pipeline actuel : 64 / 205
category
Minimum Commitment                    5
Revenue/Profit Sharing                5
Most Favored Nation                   4
Exclusivity                           3
Rofr/Rofo/Rofn                        3
Liquidated Damages                    3
Covenant Not To Sue                   3
Ip Ownership Assignment               3
Competitive Restriction Exception     3
Volume Restriction                    2
Effective Date                        2
Affiliate License-Licensor            2
Unlimited/All-You-Can-Eat-License     2
Non-Transferable License              2
Termination For Convenience           2
Non-Disparagement                     2
Non-Compete                           2
Post-Termination Services             2
Price Restrictions                    2
No-Solicit Of Customers               2
Cap On Liability                      1
Irrevocable Or Perpetual License      1
Notice Period To Terminate Renewal    1
Insurance                    

In [53]:
import random

failure_pool = merged[(merged["n_hits"] == 0) & (~merged["category"].isin(["Document Name", "Parties"]))]

random.seed(123)
sample_indices = random.sample(list(failure_pool.index), 10)
inspection_sample = failure_pool.loc[sample_indices]

positive_lookup = {(item["title"], item["question"]): item for item in positive_sample}

In [54]:
sample_fail = current_failures.sample(min(10, len(current_failures)), random_state=456)

for _, row in sample_fail.iterrows():
    item = positive_lookup[(row["title"], row["question"])]
    ground_truth = item["ground_truth"]

    print("=" * 100)
    print(f"Categorie: {row['category']}")
    print(f"Ground truth: {ground_truth}")
    print(f"Scores des chunks retenus: {row['retrieved_scores']}")
    print("-" * 100)
    for i, chunk in enumerate(row["retrieved_chunks"]):
        print(f"[Chunk {i}]\n{chunk[:300]}...\n")


Categorie: Agreement Date
Ground truth: ['October 15, 2009']
Scores des chunks retenus: [np.float32(0.051060334), np.float32(0.049580794), np.float32(0.048407525)]
----------------------------------------------------------------------------------------------------
[Chunk 0]
Effective as of the Agreement Date, PPI hereby sells, transfers, conveys and assigns to EKR all right, title and interest in and to [**] (the "Transferred NDA"). Each Party shall, within five (5) business days after the Agreement Date, file with the FDA a notice letter, substantially in the form att...

[Chunk 1]
On the Agreement Date, PPI shall execute and deliver to EKR a Bill of Sale with respect to the Transferred Equipment substantially in the form attached hereto as Exhibit 3.20(a). (b) EKR will pay PPI [**] Dollars ($[**]) for the Transferred Equipment (the "Equipment Purchase Price") as follows:

(i)...

[Chunk 2]
3.4 Assistance. PPI shall, at EKR's cost and expense, provide EKR with all assistance, informat

In [55]:
checkpoint["reranked_large_short_hit_rate"] = reranked_large_short_hit_rate
checkpoint["reranked_large_short_hit_by_category"] = reranked_large_short_hit_by_category
checkpoint["reranked_large_short_df"] = reranked_large_short_df
checkpoint["positive_large_short_scores_df"] = positive_large_short_scores_df
checkpoint["negative_large_short_scores_df"] = negative_large_short_scores_df

with open("../artifacts/baseline_v2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Baseline v2 figee et sauvegardee (0.688).")


Baseline v2 figee et sauvegardee (0.688).


In [56]:
category_to_tier

{'Agreement Date': 2,
 'Document Name': 1,
 'Effective Date': 2,
 'Parties': 1,
 'Governing Law': 2,
 'Expiration Date': 2,
 'Third Party Beneficiary': 2,
 'Termination For Convenience': 2,
 'Notice Period To Terminate Renewal': 2,
 'Renewal Term': 2,
 'Anti-Assignment': 2,
 'No-Solicit Of Customers': 2,
 'Volume Restriction': 2,
 'No-Solicit Of Employees': 2,
 'Most Favored Nation': 2,
 'Price Restrictions': 2,
 'Uncapped Liability': 2,
 'Covenant Not To Sue': 2,
 'Non-Disparagement': 2,
 'Unlimited/All-You-Can-Eat-License': 2,
 'Liquidated Damages': 2,
 'Warranty Duration': 2,
 'Competitive Restriction Exception': 2,
 'Change Of Control': 2,
 'Exclusivity': 2,
 'Joint Ip Ownership': 2,
 'Non-Compete': 2,
 'Revenue/Profit Sharing': 2,
 'Minimum Commitment': 3,
 'Non-Transferable License': 3,
 'Cap On Liability': 3,
 'Audit Rights': 3,
 'Ip Ownership Assignment': 3,
 'Insurance': 3,
 'Affiliate License-Licensee': 3,
 'Post-Termination Services': 3,
 'License Grant': 3,
 'Irrevocable Or

In [71]:
DISPERSED_RESPONSES = {"Minimum Commitment",
"Revenue/Profit Sharing",
"Most Favored Nation",
"Exclusivity",
"Rofr/Rofo/Rofn",
"Liquidated Damages",
"Covenant Not To Sue",
"Ip Ownership Assignment",
"Competitive Restriction Exception",
"Volume Restriction",
"Effective Date",
"Affiliate License-Licensor",
"Unlimited/All-You-Can-Eat-License",
"Non-Transferable License",
"Termination For Convenience",
"Non-Disparagement",
"Non-Compete",
"Post-Termination Services",
"Price Restrictions",
"No-Solicit Of Customers"}

In [72]:
def retrieve_scoped_reranked_adaptive_topk(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"]==title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if category in DISPERSED_RESPONSES:
        retrieve_top_k=15
        sparse_top_k=15
        rerank_top_n=5
    
    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [73]:
adaptive_hit_rate, adaptive_hit_by_category, adaptive_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid, retrieve_fn=retrieve_scoped_reranked_adaptive_topk
)
print(f"top_k adaptif (large loop for disperded responses): {adaptive_hit_rate:.3f}")

top_k adaptif (large loop for disperded responses): 0.746


In [74]:
adaptive_hit_by_category

category
Minimum Commitment                    0.0
Most Favored Nation                   0.2
Revenue/Profit Sharing                0.2
Exclusivity                           0.4
Competitive Restriction Exception     0.4
Covenant Not To Sue                   0.4
Liquidated Damages                    0.4
Rofr/Rofo/Rofn                        0.6
Ip Ownership Assignment               0.6
Non-Disparagement                     0.6
No-Solicit Of Customers               0.6
Unlimited/All-You-Can-Eat-License     0.6
Volume Restriction                    0.6
Post-Termination Services             0.6
Notice Period To Terminate Renewal    0.8
Non-Transferable License              0.8
Source Code Escrow                    0.8
Affiliate License-Licensee            0.8
Cap On Liability                      0.8
Affiliate License-Licensor            0.8
Agreement Date                        0.8
Joint Ip Ownership                    0.8
Effective Date                        0.8
Warranty Duration        

In [75]:
current_failures = adaptive_df[adaptive_df["hit"] == 0]
print(f"Echecs avec le pipeline actuel : {len(current_failures)} / {len(adaptive_df)}")
print(current_failures["category"].value_counts())

Echecs avec le pipeline actuel : 52 / 205
category
Minimum Commitment                    5
Most Favored Nation                   4
Revenue/Profit Sharing                4
Covenant Not To Sue                   3
Exclusivity                           3
Liquidated Damages                    3
Competitive Restriction Exception     3
Volume Restriction                    2
Unlimited/All-You-Can-Eat-License     2
Ip Ownership Assignment               2
Non-Disparagement                     2
Post-Termination Services             2
Rofr/Rofo/Rofn                        2
No-Solicit Of Customers               2
Cap On Liability                      1
Insurance                             1
Effective Date                        1
Warranty Duration                     1
Irrevocable Or Perpetual License      1
Notice Period To Terminate Renewal    1
Non-Transferable License              1
Affiliate License-Licensee            1
Joint Ip Ownership                    1
Expiration Date              

In [76]:
# 1. Reset : tous les Tier 3 actuels -> Tier 2
for category, tier in list(category_to_tier.items()):
    if tier == 3:
        category_to_tier[category] = 2

# 2. Promotion vers Tier 3 uniquement pour les categories qui ont prouve un gain avec le pool elargi
IMPROVED_WITH_WIDER_POOL = {
    "Termination For Convenience",
    "Non-Compete",
    "Price Restrictions",
    "Revenue/Profit Sharing",
    "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment",
    "Effective Date",
    "Affiliate License-Licensor",
    "Non-Transferable License",
}

for category in IMPROVED_WITH_WIDER_POOL:
    category_to_tier[category] = 3

print(category_to_tier)


{'Agreement Date': 2, 'Document Name': 1, 'Effective Date': 3, 'Parties': 1, 'Governing Law': 2, 'Expiration Date': 2, 'Third Party Beneficiary': 2, 'Termination For Convenience': 3, 'Notice Period To Terminate Renewal': 2, 'Renewal Term': 2, 'Anti-Assignment': 2, 'No-Solicit Of Customers': 2, 'Volume Restriction': 2, 'No-Solicit Of Employees': 2, 'Most Favored Nation': 2, 'Price Restrictions': 3, 'Uncapped Liability': 2, 'Covenant Not To Sue': 2, 'Non-Disparagement': 2, 'Unlimited/All-You-Can-Eat-License': 2, 'Liquidated Damages': 2, 'Warranty Duration': 2, 'Competitive Restriction Exception': 2, 'Change Of Control': 2, 'Exclusivity': 2, 'Joint Ip Ownership': 2, 'Non-Compete': 3, 'Revenue/Profit Sharing': 3, 'Minimum Commitment': 2, 'Non-Transferable License': 3, 'Cap On Liability': 2, 'Audit Rights': 2, 'Ip Ownership Assignment': 3, 'Insurance': 2, 'Affiliate License-Licensee': 2, 'Post-Termination Services': 2, 'License Grant': 2, 'Irrevocable Or Perpetual License': 2, 'Rofr/Rofo/Ro

In [80]:
def retrieve_scoped_reranked_adaptive_topk(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [90]:
from llama_index.core.node_parser import SentenceWindowNodeParser

relevant_titles = set(item["title"] for item in positive_sample)
print(f"nbre de contrats uniques concernés: {len(relevant_titles)}")

relevant_contracts = [c for c in unique_contracts if c["title"] in relevant_titles]

sentence_documents = [
    Document(text=c["clean_context"], metadata={"title": c["title"]})
    for c in relevant_contracts
]

sentence_window_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

sentence_nodes = sentence_window_parser.get_nodes_from_documents(sentence_documents)

counters = defaultdict(int)
for n in sentence_nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"]=counters[title]
    counters[title]+=1

print(f"nombre de phrases indexés: {len(sentence_nodes)}")
print(f"moy de phrases par contrat: {len(sentence_nodes)/len(relevant_contracts):.1f}")

nbre de contrats uniques concernés: 161
nombre de phrases indexés: 50675
moy de phrases par contrat: 314.8


In [91]:
vector_store_sentence = QdrantVectorStore(
    collection_name="cuad_sentence_widow_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25"
)
storage_context_sentence=StorageContext.from_defaults(vector_store=vector_store_sentence)
index_sentence=VectorStoreIndex(sentence_nodes, storage_context=storage_context_sentence, embed_model=nomic_embed)


Fetching 18 files: 100%|██████████| 18/18 [00:02<00:00,  8.94it/s]
/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/llama_index/vector_stores/qdrant/base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/qdrant_client/qdrant_client.py:1927: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  return self._client.upload_points(


In [92]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

metadata_replacer = MetadataReplacementPostProcessor(target_metadata_key="window")

def retrieve_scoped_sentence_window(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in sentence_nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.metadata.get("window", n.get_content()), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    retrieved_nodes = metadata_replacer.postprocess_nodes(retrieved_nodes)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [93]:
sentence_window_hit_rate, sentence_window_hit_by_category, sentence_window_df = compute_hit_rate_reranked(
    positive_sample, index_sentence, retrieve_fn=retrieve_scoped_sentence_window
)
print(f"Sentence-window chunking hit rate: {sentence_window_hit_rate:.3f}")


Sentence-window chunking hit rate: 0.678


In [94]:
sentence_window_hit_by_category

category
Most Favored Nation                   0.0
Exclusivity                           0.0
Minimum Commitment                    0.2
Revenue/Profit Sharing                0.2
Competitive Restriction Exception     0.2
Covenant Not To Sue                   0.2
Volume Restriction                    0.2
Price Restrictions                    0.4
Liquidated Damages                    0.4
Joint Ip Ownership                    0.4
Ip Ownership Assignment               0.4
Rofr/Rofo/Rofn                        0.6
Post-Termination Services             0.6
Non-Disparagement                     0.6
Non-Compete                           0.6
Warranty Duration                     0.6
Agreement Date                        0.6
Irrevocable Or Perpetual License      0.8
Unlimited/All-You-Can-Eat-License     0.8
Source Code Escrow                    0.8
Affiliate License-Licensor            0.8
Audit Rights                          0.8
Non-Transferable License              0.8
Cap On Liability         

In [95]:
sentence_window_parser_6 = SentenceWindowNodeParser.from_defaults(
    window_size=6,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

sentence_nodes_6 = sentence_window_parser_6.get_nodes_from_documents(sentence_documents)

counters = defaultdict(int)
for n in sentence_nodes_6:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"nombre de phrases indexees: {len(sentence_nodes_6)}")


nombre de phrases indexees: 50675


In [96]:
vector_store_sentence_6 = QdrantVectorStore(
    collection_name="cuad_sentence_window6_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25"
)
storage_context_sentence_6 = StorageContext.from_defaults(vector_store=vector_store_sentence_6)
index_sentence_6 = VectorStoreIndex(sentence_nodes_6, storage_context=storage_context_sentence_6, embed_model=nomic_embed)


In [97]:
def retrieve_scoped_sentence_window_6(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in sentence_nodes_6 if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.metadata.get("window", n.get_content()), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)
    retrieved_nodes = metadata_replacer.postprocess_nodes(retrieved_nodes)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [98]:
sentence_window6_hit_rate, sentence_window6_hit_by_category, sentence_window6_df = compute_hit_rate_reranked(
    positive_sample, index_sentence_6, retrieve_fn=retrieve_scoped_sentence_window_6
)
print(f"Sentence-window (size=6) hit rate: {sentence_window6_hit_rate:.3f}")


Sentence-window (size=6) hit rate: 0.683


In [99]:
sentence_window6_hit_by_category

category
Minimum Commitment                    0.0
Most Favored Nation                   0.0
Revenue/Profit Sharing                0.2
Volume Restriction                    0.2
Exclusivity                           0.2
Rofr/Rofo/Rofn                        0.4
Price Restrictions                    0.4
Covenant Not To Sue                   0.4
Liquidated Damages                    0.4
Joint Ip Ownership                    0.4
Ip Ownership Assignment               0.4
Unlimited/All-You-Can-Eat-License     0.6
Post-Termination Services             0.6
Non-Compete                           0.6
Warranty Duration                     0.6
Cap On Liability                      0.6
Competitive Restriction Exception     0.6
Affiliate License-Licensee            0.8
Source Code Escrow                    0.8
Insurance                             0.8
Non-Transferable License              0.8
Non-Disparagement                     0.8
Audit Rights                          0.8
No-Solicit Of Employees  

In [100]:
checkpoint["category_to_tier"] = category_to_tier
checkpoint["adaptive_hit_rate"] = adaptive_hit_rate
checkpoint["adaptive_hit_by_category"] = adaptive_hit_by_category
checkpoint["adaptive_df"] = adaptive_df

# pistes explorees et rejetees (documentation)
checkpoint["sentence_window_hit_rate"] = sentence_window_hit_rate
checkpoint["sentence_window6_hit_rate"] = sentence_window6_hit_rate

with open("../artifacts/baseline_v2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Checkpoint mis a jour - meilleure config: 0.746 (chunking fixe + hybride + bge-reranker-large + top_k adaptatif)")


Checkpoint mis a jour - meilleure config: 0.746 (chunking fixe + hybride + bge-reranker-large + top_k adaptatif)


In [102]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core.node_parser import LangchainNodeParser

lc_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    length_function=lambda text: len(tokenizer.encode(text)),
)

recursive_parser = LangchainNodeParser(lc_splitter)
recursive_nodes = recursive_parser.get_nodes_from_documents(sentence_documents)

counters = defaultdict(int)
for n in recursive_nodes:
    title=n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title]+=1

print(f"nbre de chunks (recursive): {len(recursive_nodes)}")
print(f"moy de chunks par contrat: {len(recursive_nodes)/len(relevant_contracts):.1f}")



nbre de chunks (recursive): 9776
moy de chunks par contrat: 60.7


In [103]:
vector_store_recursive= QdrantVectorStore(
    collection_name="cuad_recursive_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25"
)

storage_context_recursive = StorageContext.from_defaults(vector_store=vector_store_recursive)
index_recursive = VectorStoreIndex(recursive_nodes, storage_context=storage_context_recursive, embed_model=nomic_embed)


In [104]:
def retrieve_scoped_recursive(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in recursive_nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [105]:
recursive_hit_rate, recursive_hit_by_category, recursive_df = compute_hit_rate_reranked(
    positive_sample, index_recursive, retrieve_fn=retrieve_scoped_recursive
)
print(f"Recursive chunking hit rate: {recursive_hit_rate:.3f}")


Recursive chunking hit rate: 0.624


In [106]:
recursive_hit_by_category

category
Minimum Commitment                    0.0
Revenue/Profit Sharing                0.0
Most Favored Nation                   0.0
Ip Ownership Assignment               0.2
Volume Restriction                    0.2
Exclusivity                           0.2
Competitive Restriction Exception     0.2
Covenant Not To Sue                   0.2
Post-Termination Services             0.2
Rofr/Rofo/Rofn                        0.2
Liquidated Damages                    0.4
No-Solicit Of Customers               0.4
License Grant                         0.4
Non-Disparagement                     0.6
Non-Compete                           0.6
Price Restrictions                    0.6
Uncapped Liability                    0.6
Notice Period To Terminate Renewal    0.6
Warranty Duration                     0.6
Affiliate License-Licensor            0.6
Irrevocable Or Perpetual License      0.8
Unlimited/All-You-Can-Eat-License     0.8
Termination For Convenience           0.8
Source Code Escrow       

In [107]:
def retrieve_scoped_recursive(title, question, category, index, retrieve_top_k=20, sparse_top_k=20, rerank_top_n=6):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in recursive_nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 30
        sparse_top_k = 30
        rerank_top_n = 10

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [108]:
allrecursive_hit_rate, allrecursive_hit_by_category, allrecursive_df = compute_hit_rate_reranked(
    positive_sample, index_recursive, retrieve_fn=retrieve_scoped_recursive
)
print(f"Recursive chunking hit rate: {allrecursive_hit_rate:.3f}")


Recursive chunking hit rate: 0.624


In [109]:
import random
random.seed(789)

sample_titles = random.sample([c["title"] for c in unique_contracts], 15)

for title in sample_titles:
    contract = next(c for c in unique_contracts if c["title"] == title)
    print("=" * 100)
    print(f"TITRE: {title}")
    print("-" * 100)
    print(contract["clean_context"][:1500])
    print()


TITRE: AURASYSTEMSINC_06_16_2010-EX-10.25-STRATEGIC ALLIANCE AGREEMENT
----------------------------------------------------------------------------------------------------
STRATEGIC ALLIANCE AGREEMENT This STRATEGIC ALLIANCE AGREEMENT (the "Agreement") is entered into as of March 18, 2010 (the "Effective Date"), between AURA SYSTEMS INC., a Delaware Corporation ("Aura") and ZANOTTI EAST INC., a Massachusetts Corporation ("Zanotti"). WHEREAS, Aura has invented, manufactures and distributes a unique, integrated electromagnetic mobile power generation system capable of delivering on-demand both AC and DC power for numerous end-uses, including without limitation, all-electric transport refrigeration (the "AuraGen"); WHEREAS, the AuraGen is the subject of substantial proprietary information, including but not limited to patents, trademarks, trade secrets, know-how, and confidential information owned by Aura; WHEREAS, Zanotti is a world-wide leading manufacturer and distributor of transport 

In [110]:
checkpoint["recursive_hit_rate"] = recursive_hit_rate
checkpoint["recursive_hit_by_category"] = recursive_hit_by_category
checkpoint["recursive_df"] = recursive_df

checkpoint["sentence_window_hit_by_category"] = sentence_window_hit_by_category
checkpoint["sentence_window_df"] = sentence_window_df

checkpoint["sentence_window6_hit_by_category"] = sentence_window6_hit_by_category
checkpoint["sentence_window6_df"] = sentence_window6_df

with open("../artifacts/baseline_v2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Checkpoint complet sauvegarde avant pause.")


Checkpoint complet sauvegarde avant pause.


In [1]:
import pickle
with open("../artifacts/baseline_v2_checkpoint.pkl", "rb") as f:
    checkpoint = pickle.load(f)
for key, value in checkpoint.items():
    globals()[key] = value
print("Checkpoint recharge:", list(checkpoint.keys()))


Checkpoint recharge: ['baseline_hit_rate', 'baseline_hit_vy_category', 'baseline_df', 'bge_large_hit_rate', 'bge_large_hit_vy_category', 'bge_large_df', 'nomic_hit_rate', 'nomic_hit_vy_category', 'nomic_df', 'e5_hit_rate', 'e5_hit_by_category', 'e5_df', 'hybrid_hit_rate', 'hybrid_hit_by_category', 'hybrid_df', 'reranked_hit_rate', 'reranked_hit_by_category', 'reranked_df', 'merged', 'positive_sample', 'negative_sample', 'category_to_tier', 'reranked_large_short_hit_rate', 'reranked_large_short_hit_by_category', 'reranked_large_short_df', 'positive_large_short_scores_df', 'negative_large_short_scores_df', 'adaptive_hit_rate', 'adaptive_hit_by_category', 'adaptive_df', 'sentence_window_hit_rate', 'sentence_window6_hit_rate', 'recursive_hit_rate', 'recursive_hit_by_category', 'recursive_df', 'sentence_window_hit_by_category', 'sentence_window_df', 'sentence_window6_hit_by_category', 'sentence_window6_df']


In [2]:
from qdrant_client import QdrantClient
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex

client = QdrantClient(path="../artifacts/qdrant_storage")
nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")

vector_store_hybrid = QdrantVectorStore(
    collection_name="cuad_nomic_bm25_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
index_hybrid = VectorStoreIndex.from_vector_store(vector_store_hybrid, embed_model=nomic_embed)


/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_2599/1507995236.py:6: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_widow_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="../artifacts/qdrant_storage")
/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_2599/1507995236.py:6: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_window6_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performanc

In [8]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

22450
4182


In [9]:
full_data = concatenate_datasets([train_data, test_data])

In [10]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [11]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

In [12]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

In [13]:
for item in positive_sample:
    item["ground_truth_clean"] = [clean_contract_text(a) for a in item["ground_truth"]]

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

In [15]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from collections import defaultdict

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

Splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    tokenizer=tokenizer.encode
)

documents = [
    Document(
        text=c["clean_context"], metadata={"title": c["title"]})
        for c in unique_contracts
]

nodes = Splitter.get_nodes_from_documents(documents)

# chunk index for each contract
counters = defaultdict(int)
for n in nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"Number of contracts : {len(documents)}")
print(f"Number of chunks: {len(nodes)}")
print(f"Average chunks per contract: {len(nodes)/len(documents):.1f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7558 > 512). Running this sequence through the model will result in indexing errors


Number of contracts : 510
Number of chunks: 14230
Average chunks per contract: 27.9


In [16]:
from llama_index.core.postprocessor import SentenceTransformerRerank
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-large", top_n=3)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4725.32it/s]


In [17]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
import re

def strip_boilerplate(question):
    match = re.search(r'"([^"]+)"', question)
    category_quoted = match.group(0) if match else ""
    details = question.split("Details: ", 1)[1] if "Details: " in question else ""
    return f"{category_quoted}. {details}".strip()

def retrieve_scoped_reranked_adaptive_topk(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [18]:
from llama_index.core import PromptTemplate

qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [19]:
from llama_index.llms.ollama import Ollama
llm = Ollama(model="llama3.2:3b", request_timeout=180, temperature=0.0)


In [20]:
def generate_answer(title, question, category):
    retrieved = retrieve_scoped_reranked_adaptive_topk(title, question, category, index_hybrid)
    contexts = [r["text"] for r in retrieved]
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [21]:
from tqdm import tqdm
import json

def run_generation(sample, is_positive, output_path):
    results = []
    for item in tqdm(sample):
        try:
            answer, contexts = generate_answer(item["title"], item["question"], item["category"])
        except Exception as e:
            answer = f"[ERROR: {type(e).__name__}: {e}]"
            contexts = []

        result = {
            "title": item["title"],
            "category": item["category"],
            "tier": category_to_tier.get(item["category"], 2),
            "question": item["question"],
            "answer": answer,
            "contexts": contexts,
            "is_positive": is_positive,
        }
        if is_positive:
            result["ground_truth"] = " ".join(item["ground_truth_clean"])
        else:
            result["ground_truth"] = "This clause is not present in this contract."

        results.append(result)
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results


In [22]:
positive_results_v2 = run_generation(positive_sample, is_positive=True, output_path="../artifacts/positive_results_v2.json")
print(f"saved {len(positive_results_v2)} positive results.")

 19%|█▉        | 39/205 [13:44<1:17:43, 28.09s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xad3d0c000>
    label = <none> 
    device = <AGXG13XDevice: 0x103dd7610>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x103dd9ad0>
        label = <none> 
        device = <AGXG13XDevice: 0x103dd7610>
            name = Apple M1 Pro 
    retainedReferences = 1
 36%|███▌      | 73/205 [29:17<1:01:46, 28.08s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xad3d0c000>
    label = <none> 
    device = <AGXG13XDevice: 0x103dd7610>
        name = Apple M1 Pro 


saved 205 positive results.


In [23]:
positive_results_v2_clean = [r for r in positive_results_v2 if not r["answer"].startswith("[ERROR")]
print(f"{len(positive_results_v2_clean)} usable after removing errors ({len(positive_results_v2) - len(positive_results_v2_clean)} errors)")

ragas_data = [
    {
        "user_input": item["question"],
        "response": item["answer"],
        "retrieved_contexts": item["contexts"],
        "reference": item["ground_truth"],
    }
    for item in positive_results_v2_clean
]


205 usable after removing errors (0 errors)


In [28]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-haiku-4-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)
ragas_embeddings = LlamaIndexEmbeddingsWrapper(nomic_embed)


In [29]:
from ragas.metrics import faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig

metrics = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
run_config_claude = RunConfig(timeout=180, max_workers=4)


In [32]:
from ragas import EvaluationDataset, evaluate
import json, time, os

BATCH_SIZE = 20
output_path = "../artifacts/ragas_scores_v2.json"

if os.path.exists(output_path):
    with open(output_path) as f:
        all_scores = json.load(f)
else:
    all_scores = []

start_index = len(all_scores)
n_batches = (len(ragas_data) - 1) // BATCH_SIZE + 1

for i in range(start_index, len(ragas_data), BATCH_SIZE):
    batch = ragas_data[i : i + BATCH_SIZE]
    batch_dataset = EvaluationDataset.from_list(batch)
    print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

    result = evaluate(dataset=batch_dataset, metrics=metrics, llm=ragas_llm_claude,
                       embeddings=ragas_embeddings, run_config=run_config_claude)

    all_scores.extend(result.to_pandas().to_dict(orient="records"))
    with open(output_path, "w") as f:
        json.dump(all_scores, f, indent=2)
    print(f" -> saved ({len(all_scores)}/{len(ragas_data)})")


In [33]:
import json
import pandas as pd

with open("../artifacts/ragas_scores_v2.json") as f:
    all_scores = json.load(f)

df_check = pd.DataFrame(all_scores)
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

failed_indices = df_check[df_check[metric_cols].isna().any(axis=1)].index.tolist()
print(f"Nombre d'exemples a refaire: {len(failed_indices)}")


Nombre d'exemples a refaire: 22


In [35]:
retry_data = [
    {
        "user_input": all_scores[i]["user_input"],
        "response": all_scores[i]["response"],
        "retrieved_contexts": all_scores[i]["retrieved_contexts"],
        "reference": all_scores[i]["reference"],
    }
    for i in failed_indices
]

retry_dataset = EvaluationDataset.from_list(retry_data)
retry_result = evaluate(
    dataset=retry_dataset,
    metrics=metrics,
    llm=ragas_llm_claude,
    embeddings=ragas_embeddings,
    run_config=run_config_claude,
)
retry_scores = retry_result.to_pandas().to_dict(orient="records")


Evaluating: 100%|██████████| 110/110 [05:12<00:00,  2.84s/it]


In [36]:
for idx, new_row in zip(failed_indices, retry_scores):
    all_scores[idx].update(new_row)

with open("../artifacts/ragas_scores_v2.json", "w") as f:
    json.dump(all_scores, f, indent=2)

df_check2 = pd.DataFrame(all_scores)
print("NaN restants par metrique:")
print(df_check2[metric_cols].isna().sum())


NaN restants par metrique:
faithfulness          0
answer_relevancy      0
context_precision     0
context_recall        0
answer_correctness    2
dtype: int64


In [37]:
for score_row, meta_row in zip(all_scores, positive_results_v2_clean):
    score_row["category"] = meta_row["category"]
    score_row["tier"] = meta_row["tier"]

df_scores_v2 = pd.DataFrame(all_scores)
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

summary_global_v2 = df_scores_v2[metric_cols].mean().to_frame(name="moyenne_v2")
print(summary_global_v2)

summary_by_category_v2 = df_scores_v2.groupby("category")[metric_cols].mean()
summary_global_v2.to_csv("../artifacts/ragas_scores_v2_summary_global.csv")
summary_by_category_v2.to_csv("../artifacts/ragas_scores_v2_summary_by_category.csv")


                    moyenne_v2
faithfulness          0.784636
answer_relevancy      0.573797
context_precision     0.366064
context_recall        0.474936
answer_correctness    0.393670


In [38]:
df_scores_v2["was_widened"] = df_scores_v2["category"].apply(lambda c: category_to_tier.get(c, 2) == 3)
comparison = df_scores_v2.groupby("was_widened")[metric_cols].mean()
print(comparison)


             faithfulness  answer_relevancy  context_precision  \
was_widened                                                      
False            0.785594          0.574299           0.377604   
True             0.781230          0.572014           0.325031   

             context_recall  answer_correctness  
was_widened                                      
False              0.445699            0.401436  
True               0.578889            0.366404  


In [39]:
def extract_all_chunk_scores(sample, index, retrieve_fn=retrieve_scoped_reranked_adaptive_topk):
    rows = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        tier = category_to_tier.get(category, 2)

        retrieved = retrieve_fn(title, question, category, index)

        for position, r in enumerate(retrieved, start=1):
            rows.append({
                "title": title,
                "category": category,
                "tier": tier,
                "position": position,
                "score": r["score"],
            })
    return pd.DataFrame(rows)


In [40]:
positive_chunk_scores_df = extract_all_chunk_scores(positive_sample, index_hybrid)
positive_chunk_scores_df["is_positive"] = True

negative_chunk_scores_df = extract_all_chunk_scores(negative_sample, index_hybrid)
negative_chunk_scores_df["is_positive"] = False

all_chunk_scores_df = pd.concat([positive_chunk_scores_df, negative_chunk_scores_df], ignore_index=True)


In [41]:
all_chunk_scores_df.to_pickle("../artifacts/all_chunk_scores_v2.pkl")
print(f"Sauvegarde : {len(all_chunk_scores_df)} lignes (une par chunk x position)")


Sauvegarde : 1074 lignes (une par chunk x position)


In [42]:
summary_by_position = (
    all_chunk_scores_df.dropna(subset=["score"])
    .groupby(["position", "is_positive"])["score"]
    .describe()
)
print(summary_by_position)


                      count      mean       std       min       25%       50%  \
position is_positive                                                            
1        False        117.0  0.005376  0.014560  0.000077  0.000240  0.000813   
         True         195.0  0.054134  0.127143  0.000100  0.000742  0.003682   
2        False        114.0  0.001939  0.005140  0.000082  0.000191  0.000441   
         True         194.0  0.009234  0.032343  0.000084  0.000385  0.000902   
3        False        111.0  0.001095  0.002249  0.000079  0.000152  0.000348   
         True         194.0  0.003976  0.013556  0.000084  0.000275  0.000533   
4        False         24.0  0.000487  0.000537  0.000078  0.000131  0.000352   
         True          45.0  0.003606  0.008892  0.000081  0.000185  0.000374   
5        False         24.0  0.000352  0.000386  0.000077  0.000121  0.000223   
         True          45.0  0.002329  0.005977  0.000081  0.000157  0.000321   

                           

In [43]:
POSITION_THRESHOLDS = {
    2: 0.00029,
    3: 0.00025,
    4: 0.00066,
    5: 0.00096,
}

def retrieve_scoped_reranked_threshold(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    kept = []
    for position, n in enumerate(reranked_nodes, start=1):
        if position == 1:
            kept.append({"text": n.get_content(), "score": n.score})
        elif n.score is not None and n.score >= POSITION_THRESHOLDS.get(position, float("inf")):
            kept.append({"text": n.get_content(), "score": n.score})

    return kept


In [46]:
def compute_hit_rate_reranked(sample, index, retrieve_fn=retrieve_scoped_reranked_threshold, rerank_top_n=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        reference_answers = item["ground_truth"]

        retrieved = retrieve_fn(title, question, category, index, rerank_top_n=rerank_top_n)
        retrieved_texts = [r["text"] for r in retrieved]
        retrieved_scores = [r["score"] for r in retrieved]

        hit = is_hit(reference_answers, retrieved_texts)

        results.append({
            "title": title,
            "category": category,
            "question": question,
            "hit": hit,
            "retrieved_chunks": retrieved_texts,
            "retrieved_scores": retrieved_scores,
        })

    df_hits = pd.DataFrame(results)
    overall_hit_rate = df_hits["hit"].mean()
    hit_rate_by_category = df_hits.groupby("category")["hit"].mean().sort_values()
    return overall_hit_rate, hit_rate_by_category, df_hits


In [48]:
import re
import pandas as pd

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def is_hit(reference_answers, retrieved_chunks):
    normalized_chunks = [normalize_text(chunk) for chunk in retrieved_chunks]
    for ref in reference_answers:
        normalized_ref = normalize_text(ref)
        if not normalized_ref:
            continue
        for chunk in normalized_chunks:
            if normalized_ref in chunk:
                return 1
        
    return 0

def compute_hit_rate(sample, index, top_k=3):
    results = []
    for item in sample:
        title = item["title"]
        question = item["question"]
        category = item["category"]
        reference_answers = item["ground_truth"]

        retrieved_chunks = retrieve_scoped_generic(title, question, category, index, top_k=top_k)
        hit = is_hit(reference_answers, retrieved_chunks)

        results.append({"title": title, "category": category, "question": question, "hit": hit})

    
    df_hits = pd.DataFrame(results)
    overall_hit_rate = df_hits["hit"].mean()
    hit_rate_by_category = df_hits.groupby("category")["hit"].mean().sort_values()
    return overall_hit_rate, hit_rate_by_category, df_hits
    

In [49]:
threshold_hit_rate, threshold_hit_by_category, threshold_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid, retrieve_fn=retrieve_scoped_reranked_threshold
)
print(f"Retrieval avec filtrage par seuil: {threshold_hit_rate:.3f}")


Retrieval avec filtrage par seuil: 0.663


In [50]:
print(threshold_hit_by_category)


category
Minimum Commitment                    0.0
Revenue/Profit Sharing                0.0
Rofr/Rofo/Rofn                        0.2
Most Favored Nation                   0.2
Exclusivity                           0.2
Volume Restriction                    0.4
No-Solicit Of Customers               0.4
Non-Compete                           0.4
Covenant Not To Sue                   0.4
Competitive Restriction Exception     0.4
Price Restrictions                    0.4
Ip Ownership Assignment               0.4
Liquidated Damages                    0.4
Non-Disparagement                     0.6
Non-Transferable License              0.6
Post-Termination Services             0.6
Unlimited/All-You-Can-Eat-License     0.6
Warranty Duration                     0.6
Cap On Liability                      0.8
Affiliate License-Licensor            0.8
Termination For Convenience           0.8
Source Code Escrow                    0.8
Agreement Date                        0.8
Audit Rights             

In [51]:
comparison_hit = pd.DataFrame({
    "sans_filtrage_0.746": adaptive_hit_by_category,
    "avec_filtrage_0.663": threshold_hit_by_category,
})
comparison_hit["delta"] = comparison_hit["avec_filtrage_0.663"] - comparison_hit["sans_filtrage_0.746"]
print(comparison_hit.sort_values("delta").to_string())


                                    sans_filtrage_0.746  avec_filtrage_0.663  delta
category                                                                           
Price Restrictions                                  1.0                  0.4   -0.6
Non-Compete                                         1.0                  0.4   -0.6
Rofr/Rofo/Rofn                                      0.6                  0.2   -0.4
Non-Transferable License                            0.8                  0.6   -0.2
Warranty Duration                                   0.8                  0.6   -0.2
Revenue/Profit Sharing                              0.2                  0.0   -0.2
Exclusivity                                         0.4                  0.2   -0.2
Termination For Convenience                         1.0                  0.8   -0.2
Ip Ownership Assignment                             0.6                  0.4   -0.2
Audit Rights                                        1.0                  0.8

In [52]:
POSITION_THRESHOLDS_SOFT = {
    2: 0.00029,
    3: 0.00020,
    4: 0.00012,
    5: 0.00011,
}

def retrieve_scoped_reranked_threshold_soft(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    kept = []
    for position, n in enumerate(reranked_nodes, start=1):
        if position == 1:
            kept.append({"text": n.get_content(), "score": n.score})
        elif n.score is not None and n.score >= POSITION_THRESHOLDS_SOFT.get(position, float("inf")):
            kept.append({"text": n.get_content(), "score": n.score})

    return kept


In [53]:
threshold_soft_hit_rate, threshold_soft_hit_by_category, threshold_soft_df = compute_hit_rate_reranked(
    positive_sample, index_hybrid, retrieve_fn=retrieve_scoped_reranked_threshold_soft
)
print(f"Retrieval avec filtrage seuil doux: {threshold_soft_hit_rate:.3f}")


Retrieval avec filtrage seuil doux: 0.693


In [54]:
comparison_hit_v2 = pd.DataFrame({
    "sans_filtrage_0.746": adaptive_hit_by_category,
    "seuil_strict_0.663": threshold_hit_by_category,
    "seuil_doux_0.693": threshold_soft_hit_by_category,
})
comparison_hit_v2["delta_doux_vs_sans"] = comparison_hit_v2["seuil_doux_0.693"] - comparison_hit_v2["sans_filtrage_0.746"]
print(comparison_hit_v2.sort_values("delta_doux_vs_sans").to_string())


                                    sans_filtrage_0.746  seuil_strict_0.663  seuil_doux_0.693  delta_doux_vs_sans
category                                                                                                         
Non-Compete                                         1.0                 0.4               0.4                -0.6
Price Restrictions                                  1.0                 0.4               0.6                -0.4
Warranty Duration                                   0.8                 0.6               0.6                -0.2
Volume Restriction                                  0.6                 0.4               0.4                -0.2
Audit Rights                                        1.0                 0.8               0.8                -0.2
Rofr/Rofo/Rofn                                      0.6                 0.2               0.4                -0.2
Ip Ownership Assignment                             0.6                 0.4             

In [55]:
def generate_answer(title, question, category):
    retrieved = retrieve_scoped_reranked_threshold_soft(title, question, category, index_hybrid)
    contexts = [r["text"] for r in retrieved]
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [57]:
checkpoint["threshold_hit_rate"] = threshold_hit_rate
checkpoint["threshold_hit_by_category"] = threshold_hit_by_category
checkpoint["threshold_df"] = threshold_df
checkpoint["threshold_soft_hit_rate"] = threshold_soft_hit_rate
checkpoint["threshold_soft_hit_by_category"] = threshold_soft_hit_by_category
checkpoint["threshold_soft_df"] = threshold_soft_df
checkpoint["POSITION_THRESHOLDS"] = POSITION_THRESHOLDS
checkpoint["POSITION_THRESHOLDS_SOFT"] = POSITION_THRESHOLDS_SOFT

with open("../artifacts/baseline_v2_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f)

print("Checkpoint mis a jour avant redemarrage.")


Checkpoint mis a jour avant redemarrage.


In [1]:
import pickle
with open("../artifacts/baseline_v2_checkpoint.pkl", "rb") as f:
    checkpoint = pickle.load(f)
for key, value in checkpoint.items():
    globals()[key] = value
print("Checkpoint recharge:", list(checkpoint.keys()))


Checkpoint recharge: ['baseline_hit_rate', 'baseline_hit_vy_category', 'baseline_df', 'bge_large_hit_rate', 'bge_large_hit_vy_category', 'bge_large_df', 'nomic_hit_rate', 'nomic_hit_vy_category', 'nomic_df', 'e5_hit_rate', 'e5_hit_by_category', 'e5_df', 'hybrid_hit_rate', 'hybrid_hit_by_category', 'hybrid_df', 'reranked_hit_rate', 'reranked_hit_by_category', 'reranked_df', 'merged', 'positive_sample', 'negative_sample', 'category_to_tier', 'reranked_large_short_hit_rate', 'reranked_large_short_hit_by_category', 'reranked_large_short_df', 'positive_large_short_scores_df', 'negative_large_short_scores_df', 'adaptive_hit_rate', 'adaptive_hit_by_category', 'adaptive_df', 'sentence_window_hit_rate', 'sentence_window6_hit_rate', 'recursive_hit_rate', 'recursive_hit_by_category', 'recursive_df', 'sentence_window_hit_by_category', 'sentence_window_df', 'sentence_window6_hit_by_category', 'sentence_window6_df', 'threshold_hit_rate', 'threshold_hit_by_category', 'threshold_df', 'threshold_soft_h

In [3]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

22450
4182


In [4]:
full_data = concatenate_datasets([train_data, test_data])

In [5]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [6]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

In [7]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

In [8]:
for item in positive_sample:
    item["ground_truth_clean"] = [clean_contract_text(a) for a in item["ground_truth"]]

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

In [10]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from collections import defaultdict

CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

Splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    tokenizer=tokenizer.encode
)

documents = [
    Document(
        text=c["clean_context"], metadata={"title": c["title"]})
        for c in unique_contracts
]

nodes = Splitter.get_nodes_from_documents(documents)

# chunk index for each contract
counters = defaultdict(int)
for n in nodes:
    title = n.metadata["title"]
    n.metadata["chunk_index"] = counters[title]
    counters[title] += 1

print(f"Number of contracts : {len(documents)}")
print(f"Number of chunks: {len(nodes)}")
print(f"Average chunks per contract: {len(nodes)/len(documents):.1f}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7558 > 512). Running this sequence through the model will result in indexing errors


Number of contracts : 510
Number of chunks: 14230
Average chunks per contract: 27.9


In [11]:
from llama_index.core.postprocessor import SentenceTransformerRerank
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-large", top_n=3)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5274.32it/s]


In [12]:
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
import re

def strip_boilerplate(question):
    match = re.search(r'"([^"]+)"', question)
    category_quoted = match.group(0) if match else ""
    details = question.split("Details: ", 1)[1] if "Details: " in question else ""
    return f"{category_quoted}. {details}".strip()


In [13]:
POSITION_THRESHOLDS_SOFT = {2: 0.00029, 3: 0.00020, 4: 0.00012, 5: 0.00011}

def retrieve_scoped_reranked_threshold_soft(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    kept = []
    for position, n in enumerate(reranked_nodes, start=1):
        if position == 1:
            kept.append({"text": n.get_content(), "score": n.score})
        elif n.score is not None and n.score >= POSITION_THRESHOLDS_SOFT.get(position, float("inf")):
            kept.append({"text": n.get_content(), "score": n.score})

    return kept


In [14]:
from llama_index.core import PromptTemplate

qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


from llama_index.llms.ollama import Ollama
llm = Ollama(model="llama3.2:3b", request_timeout=180, temperature=0.0)


In [15]:
def generate_answer(title, question, category):
    retrieved = retrieve_scoped_reranked_threshold_soft(title, question, category, index_hybrid)
    contexts = [r["text"] for r in retrieved]
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [16]:
import os
import json
from tqdm import tqdm

def run_generation(sample, is_positive, output_path):
    if os.path.exists(output_path):
        with open(output_path) as f:
            results = json.load(f)
        done_keys = {(r["title"], r["question"]) for r in results}
        remaining = [item for item in sample if (item["title"], item["question"]) not in done_keys]
        print(f"Reprise : {len(results)} deja faits, {len(remaining)} restants")
    else:
        results = []
        remaining = sample

    for item in tqdm(remaining):
        try:
            answer, contexts = generate_answer(item["title"], item["question"], item["category"])
        except Exception as e:
            answer = f"[ERROR: {type(e).__name__}: {e}]"
            contexts = []

        result = {
            "title": item["title"], "category": item["category"],
            "tier": category_to_tier.get(item["category"], 2),
            "question": item["question"], "answer": answer,
            "contexts": contexts, "is_positive": is_positive,
        }
        result["ground_truth"] = " ".join(item["ground_truth_clean"]) if is_positive else "This clause is not present in this contract."
        results.append(result)
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results


In [2]:
from qdrant_client import QdrantClient
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex

client = QdrantClient(path="../artifacts/qdrant_storage")
nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")

vector_store_hybrid = QdrantVectorStore(
    collection_name="cuad_nomic_bm25_hybrid",
    client=client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
index_hybrid = VectorStoreIndex.from_vector_store(vector_store_hybrid, embed_model=nomic_embed)


/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_20471/1507995236.py:6: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_widow_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="../artifacts/qdrant_storage")
/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_20471/1507995236.py:6: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_window6_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performa

In [17]:
positive_results_v2_threshold = run_generation(
    positive_sample, is_positive=True, output_path="../artifacts/positive_results_v2_threshold.json"
)
print(f"saved {len(positive_results_v2_threshold)} positive results.")



Reprise : 41 deja faits, 164 restants


  7%|▋         | 11/164 [04:39<1:19:27, 31.16s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xa9fbc0000>
    label = <none> 
    device = <AGXG13XDevice: 0x10183a6f0>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x10183cbb0>
        label = <none> 
        device = <AGXG13XDevice: 0x10183a6f0>
            name = Apple M1 Pro 
    retainedReferences = 1
 16%|█▌        | 26/164 [12:04<1:06:44, 29.02s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xa9fbc0000>
    label = <none> 
    device = <AGXG13XDevice: 0x10183a6f0>
        name = Apple M1 Pro 


saved 205 positive results.


In [18]:
positive_results_v2_threshold_clean = [r for r in positive_results_v2_threshold if not r["answer"].startswith("[ERROR")]
print(f"{len(positive_results_v2_threshold_clean)} usable after removing errors")

ragas_data = [
    {
        "user_input": item["question"],
        "response": item["answer"],
        "retrieved_contexts": item["contexts"],
        "reference": item["ground_truth"],
    }
    for item in positive_results_v2_threshold_clean
]


203 usable after removing errors


In [19]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-haiku-4-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)
ragas_embeddings = LlamaIndexEmbeddingsWrapper(nomic_embed)

from ragas.metrics import faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig

metrics = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
run_config_claude = RunConfig(timeout=180, max_workers=4)


In [20]:
from ragas import EvaluationDataset, evaluate
import json, time, os

BATCH_SIZE = 20
output_path = "../artifacts/ragas_scores_v2_threshold.json"

if os.path.exists(output_path):
    with open(output_path) as f:
        all_scores = json.load(f)
else:
    all_scores = []

start_index = len(all_scores)
n_batches = (len(ragas_data) - 1) // BATCH_SIZE + 1

for i in range(start_index, len(ragas_data), BATCH_SIZE):
    batch = ragas_data[i : i + BATCH_SIZE]
    batch_dataset = EvaluationDataset.from_list(batch)
    print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

    result = evaluate(dataset=batch_dataset, metrics=metrics, llm=ragas_llm_claude,
                       embeddings=ragas_embeddings, run_config=run_config_claude)

    all_scores.extend(result.to_pandas().to_dict(orient="records"))
    with open(output_path, "w") as f:
        json.dump(all_scores, f, indent=2)
    print(f" -> saved ({len(all_scores)}/{len(ragas_data)})")


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:50<00:00,  1.10s/it]


 -> saved (20/203)
batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [02:34<00:00,  1.55s/it]


 -> saved (40/203)
batch 3/11 (20 examples)...


...

 -> saved (180/203)
batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [04:13<00:00,  2.53s/it]


 -> saved (200/203)
batch 11/11 (3 examples)...


Evaluating: 100%|██████████| 15/15 [00:34<00:00,  2.33s/it]

 -> saved (203/203)


In [21]:
import pandas as pd

for score_row, meta_row in zip(all_scores, positive_results_v2_threshold_clean):
    score_row["category"] = meta_row["category"]

df_scores_v2t = pd.DataFrame(all_scores)
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]

summary_v2_threshold = df_scores_v2t[metric_cols].mean().to_frame(name="moyenne_v2_seuil_doux")
print(summary_v2_threshold)

summary_v2_threshold.to_csv("../artifacts/ragas_scores_v2_threshold_summary_global.csv")


                    moyenne_v2_seuil_doux
faithfulness                     0.767345
answer_relevancy                 0.578034
context_precision                0.377709
context_recall                   0.433623
answer_correctness               0.391249


In [22]:
def generate_answer(title, question, category):
    retrieved = retrieve_scoped_reranked_adaptive_topk(title, question, category, index_hybrid)
    contexts = [r["text"] for r in retrieved]
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [25]:
def retrieve_scoped_reranked_adaptive_topk(title, question, category, index, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    tier = category_to_tier.get(category, 2)
    contract_nodes = [n for n in nodes if n.metadata["title"] == title]

    if tier == 1:
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        selected = contract_nodes[:1]
        return [{"text": n.get_content(), "score": None} for n in selected]

    if tier == 3:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filtered_retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    )
    retrieved_nodes = filtered_retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [{"text": n.get_content(), "score": n.score} for n in reranked_nodes]


In [26]:
negative_results_v2 = run_generation(
    negative_sample, is_positive=False, output_path="../artifacts/negative_results_v2.json"
)
print(f"saved {len(negative_results_v2)} negative results.")


 36%|███▋      | 43/118 [22:19<40:05, 32.07s/it]  Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xa9fbc0000>
    label = <none> 
    device = <AGXG13XDevice: 0x10183a6f0>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x10183cbb0>
        label = <none> 
        device = <AGXG13XDevice: 0x10183a6f0>
            name = Apple M1 Pro 
    retainedReferences = 1
 53%|█████▎    | 62/118 [32:20<23:10, 24.83s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xa9fbc0000>
    label = <none> 
    device = <AGXG13XDevice: 0x10183a6f0>
        name = Apple M1 Pro 
  

saved 118 negative results.


In [27]:
from collections import Counter

ABSENCE_PHRASES = [
    "not present", "not specified", "not mentioned", "does not contain",
    "no clause", "not applicable", "not included", "not found", "n/a", "absent"
]

def classify_negative(answer):
    if answer.startswith("[ERROR"):
        return "error"
    answer_lower = answer.lower()
    if any(phrase in answer_lower for phrase in ABSENCE_PHRASES):
        return "true negative"
    return "false positive"

for item in negative_results_v2:
    item["classification"] = classify_negative(item["answer"])

counts_v2 = Counter(item["classification"] for item in negative_results_v2)
print(counts_v2)

total_classifiable_v2 = counts_v2["true negative"] + counts_v2["false positive"]
fp_rate_v2 = counts_v2["false positive"] / total_classifiable_v2 * 100
print(f"False positive rate v2: {fp_rate_v2:.1f}% ({counts_v2['false positive']}/{total_classifiable_v2})")


Counter({'false positive': 78, 'true negative': 40})
False positive rate v2: 66.1% (78/118)
